## 面试问题

循环历史滚动窗口：保留/丢弃哪些步、按什么信号？

## 回答主线

history 无限增长会爆预算，滚动窗口决定喂什么。纯 recency（只留最近 K 步）会丢早期硬约束；正确做法是「最近窗口 + 钉住的关键锚点」。本 Notebook 让第 1 步声明预算上限=100，之后多步，对比纯 recency 窗口（丢约束、错误批准 150）与钉住窗口（保留约束、正确拒绝 150）。

## 真实案例

多步退款任务：第 1 步用户声明预算上限=100（硬约束、应钉住），第 4 步计算出退款金额=150。对比两种窗口对「是否批准 150」的影响。数据为教学历史，不代表真实系统。

In [1]:
history = [  # 构造一段多步循环历史。
    {"step": 1, "fact": "预算上限=100", "pinned": True},  # 早期硬约束应被钉住。
    {"step": 2, "fact": "查询用户订单", "pinned": False},  # 普通中间步。
    {"step": 3, "fact": "用户确认收货地址", "pinned": False},  # 普通中间步。
    {"step": 4, "fact": "计算退款金额=150", "pinned": False},  # 最近的候选动作。
]  # 结束历史定义。

print("历史步数:", len(history))  # 展示历史长度。
for h in history:  # 逐条打印历史。
    print("  ", h)  # 展示每步内容与是否钉住。

历史步数: 4
   {'step': 1, 'fact': '预算上限=100', 'pinned': True}
   {'step': 2, 'fact': '查询用户订单', 'pinned': False}
   {'step': 3, 'fact': '用户确认收货地址', 'pinned': False}
   {'step': 4, 'fact': '计算退款金额=150', 'pinned': False}


## 基线（Baseline）

反面基线：纯 recency 窗口只保留最近 2 步。它丢掉了第 1 步的预算约束，后续决策看不到「预算上限=100」。

In [2]:
def rolling_window(history, k=2):  # 纯 recency 窗口：只保留最近 k 步。
    return history[-k:]  # 返回末尾 k 步。

recency_view = rolling_window(history, k=2)  # 取最近 2 步作为上下文。
recency_facts = [h["fact"] for h in recency_view]  # 抽取窗口内事实。
print("纯 recency 窗口:", recency_facts)  # 展示窗口丢掉了早期预算约束。
print("窗口是否含预算约束:", any("预算" in f for f in recency_facts))  # 展示预算约束已被丢弃。

纯 recency 窗口: ['用户确认收货地址', '计算退款金额=150']
窗口是否含预算约束: False


## 失败案例与修正

纯 recency 看不到预算约束，会错误批准 150 的退款。修正是钉住窗口：最近 K 步 + 所有 `pinned` 锚点，无论锚点多早都保留。

In [3]:
def pinned_window(history, k=2):  # 钉住锚点的窗口：最近 k 步加所有 pinned 步。
    recent = history[-k:]  # 取最近 k 步。
    pins = [h for h in history if h["pinned"]]  # 取所有钉住的锚点。
    seen = set()  # 用于按步号去重。
    merged = []  # 合并结果。
    for h in pins + recent:  # 锚点在前最近步在后。
        if h["step"] not in seen:  # 按步号去重。
            seen.add(h["step"])  # 记录已加入。
            merged.append(h)  # 加入合并视图。
    return merged  # 返回合并窗口。

pinned_view = pinned_window(history, k=2)  # 取钉住锚点的窗口。
pinned_facts = [h["fact"] for h in pinned_view]  # 抽取窗口内事实。
print("钉住窗口:", pinned_facts)  # 展示预算约束被保留。
print("窗口是否含预算约束:", any("预算" in f for f in pinned_facts))  # 展示约束仍在。

钉住窗口: ['预算上限=100', '用户确认收货地址', '计算退款金额=150']
窗口是否含预算约束: True


In [4]:
def approve_refund(window_facts, amount=150):  # 依据窗口内可见约束决定是否批准退款。
    budget = None  # 预置预算上限。
    for f in window_facts:  # 遍历窗口事实寻找预算约束。
        if "预算上限=" in f:  # 命中预算约束。
            budget = int(f.split("=")[1])  # 解析预算数值。
    if budget is not None and amount > budget:  # 超预算应拒绝。
        return "rejected"  # 拒绝超预算退款。
    return "approved"  # 未见约束或未超则批准。

recency_decision = approve_refund(recency_facts)  # 纯 recency 窗口下的决策。
pinned_decision = approve_refund(pinned_facts)  # 钉住窗口下的决策。
print("纯 recency 决策(150元):", recency_decision, "(看不到预算约束)")  # 展示丢约束导致错误批准。
print("钉住窗口决策(150元):", pinned_decision, "(看到预算=100)")  # 展示保留约束正确拒绝。

纯 recency 决策(150元): approved (看不到预算约束)
钉住窗口决策(150元): rejected (看到预算=100)


## 结果解读

纯 recency 窗口丢掉了「预算上限=100」，错误批准 150；钉住窗口始终保留该锚点，正确拒绝。信号不是新旧而是「后续决策是否仍依赖此事实」。锚点应在写入时打标，窗口大小与 token 预算联动。

In [5]:
print("纯 recency 保留预算约束:", any("预算" in f for f in recency_facts))  # 纯 recency 丢约束。
print("钉住窗口保留预算约束:", any("预算" in f for f in pinned_facts))  # 钉住窗口保留约束。
print("决策差异 recency vs pinned:", recency_decision, "vs", pinned_decision)  # 展示两种窗口决策不同。

纯 recency 保留预算约束: False
钉住窗口保留预算约束: True
决策差异 recency vs pinned: approved vs rejected


In [6]:
assert any("预算" in f for f in pinned_facts)  # 钉住窗口必须保留预算约束。
assert not any("预算" in f for f in recency_facts)  # 纯 recency 窗口丢掉了预算约束。
assert recency_decision == "approved"  # 纯 recency 因看不到约束错误批准。
assert pinned_decision == "rejected"  # 钉住窗口因保留约束正确拒绝。
assert len(pinned_view) >= len(recency_view)  # 钉住窗口至少包含最近窗口的信息量。
print("全部不变量通过")  # 输出测试通过信号。

全部不变量通过
